# Project2 eda inf stats tableau

## Import, load and concatenate web data

In [2]:
import pandas as pd
import numpy as np

df_clients = pd.read_csv('https://raw.githubusercontent.com/bibianglez/project2_eda_inf_stats_tableau/refs/heads/main/data/raw/df_final_experiment_clients.txt')
df_web_1 = pd.read_csv('https://raw.githubusercontent.com/bibianglez/project2_eda_inf_stats_tableau/refs/heads/main/data/raw/df_final_web_data_pt_1.txt')
df_web_2 = pd.read_csv('https://raw.githubusercontent.com/bibianglez/project2_eda_inf_stats_tableau/refs/heads/main/data/raw/df_final_web_data_pt_2.txt')
print(f'Shape df_web_1:{df_web_1.shape}')
print(f'Shape df_web_2:{df_web_2.shape}')

Shape df_web_1:(343141, 5)
Shape df_web_2:(412264, 5)


In [3]:
df_final_1_2 = pd.concat([df_web_1, df_web_2], ignore_index=True)
print(f'Shape df_final_1_2: {df_final_1_2.shape}')
df_final_1_2.head()

Shape df_final_1_2: (755405, 5)


,client_id,visitor_id,visit_id,process_step,date_time
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04


## 2. Clean df_final_1_2

In [4]:
# Lowercase column names and replace spaces with '_'
# Convert date_time to datetime
df_final_1_2.columns = (
    df_final_1_2.columns
    .str.lower()
    .str.strip()
    .str.replace(' ', '_', regex=False)
)

df_final_1_2['date_time'] = pd.to_datetime(df_final_1_2['date_time'])

print(df_final_1_2.dtypes)



client_id                int64
visitor_id                 str
visit_id                   str
process_step               str
date_time       datetime64[us]
dtype: object


## 3. Remove rows with the same consecutive `process_step`

In [5]:
# Sort by visit_id and date
# Detect consecutive duplicates within each visit_id
df_final_1_2 = df_final_1_2.sort_values(['visit_id', 'date_time']).reset_index(drop=True)
grp = df_final_1_2.groupby('visit_id')['process_step']
prev_step = grp.shift(1)   
next_step = grp.shift(-1)  

mask_duplicate = (
    (df_final_1_2['process_step'] == prev_step) |
    (df_final_1_2['process_step'] == next_step)
)

df_final_1_2 = df_final_1_2[~mask_duplicate].reset_index(drop=True)
rows_after = len(df_final_1_2)

print(f'Remaining rows:  {rows_after}')
df_final_1_2.head()

Remaining rows:  610658


,client_id,visitor_id,visit_id,process_step,date_time
0,9056452,306992881_89423906595,1000165_4190026492_760066,start,2017-06-04 01:07:29
1,9056452,306992881_89423906595,1000165_4190026492_760066,step_1,2017-06-04 01:07:32
2,9056452,306992881_89423906595,1000165_4190026492_760066,step_2,2017-06-04 01:07:56
3,9056452,306992881_89423906595,1000165_4190026492_760066,step_3,2017-06-04 01:09:13
4,9056452,306992881_89423906595,1000165_4190026492_760066,confirm,2017-06-04 01:09:50


## 4. Merge with df_final_experiment_clients → df_merged_all

We keep only `client_id` values present in the experiment clients file.

In [13]:
# Normalise column names
# Merge: df_final_1_2 layout + 'variation' column from clients

df_clients.columns = (
    df_clients.columns
    .str.lower()
    .str.strip()
    .str.replace(' ', '_', regex=False)
)

df_merged_all = df_final_1_2.merge(df_clients, on='client_id', how='inner')

print(f'Shape df_merged_all:{df_merged_all.shape}')
print(f'Unique client_id:{df_merged_all["client_id"].nunique()}')
print(f'data type:   {df_merged_all.dtypes}')
df_merged_all.head(5)

Shape df_merged_all:(365546, 6)
Unique client_id:67734
data type:   client_id                int64
visitor_id                 str
visit_id                   str
process_step               str
date_time       datetime64[us]
variation                  str
dtype: object


,client_id,visitor_id,visit_id,process_step,date_time,variation
0,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,Test
1,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,Test
2,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,Test
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:22:04,Test
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:22:08,Test


### 5.1 client_id with more than one visit_id

In [8]:
visits_per_client = (
    df_merged_all.groupby('client_id')['visit_id']
    .nunique()
    .reset_index()
    .rename(columns={'visit_id': 'num_visits'})
)

multi_visit_clients = visits_per_client[visits_per_client['num_visits'] > 1]

print(f'client_id with more than 1 visit_id: {len(multi_visit_clients)}')
multi_visit_clients.head(5)

client_id with more than 1 visit_id: 15679


,client_id,num_visits
5,1104,2
6,1186,2
12,1516,2
14,1643,3
15,1677,2


### 5.2 client_id with the full process completed (start → confirm)

In [9]:
FULL_STEPS = {'start', 'step_1', 'step_2', 'step_3', 'confirm'}

steps_per_client = (
    df_merged_all.groupby('client_id')['process_step']
    .apply(set)
    .reset_index()
    .rename(columns={'process_step': 'steps'})
)

completed_clients = steps_per_client[
    steps_per_client['steps'].apply(lambda s: FULL_STEPS.issubset(s))
]

print(f'client_id with full process (start → confirm): {len(completed_clients)}')
completed_clients.head(10)

client_id with full process (start → confirm): 38851


,client_id,steps
0,169,"{step_1, step_2, start, step_3, confirm}"
1,555,"{step_1, step_2, start, step_3, confirm}"
2,647,"{step_1, step_2, start, step_3, confirm}"
3,722,"{step_1, step_2, start, step_3, confirm}"
7,1195,"{step_1, step_2, start, step_3, confirm}"
8,1197,"{step_1, step_2, start, step_3, confirm}"
9,1336,"{step_1, step_2, start, step_3, confirm}"
12,1516,"{step_1, step_2, start, step_3, confirm}"
14,1643,"{step_1, step_2, start, step_3, confirm}"
15,1677,"{step_1, step_2, start, step_3, confirm}"


### 5.3 Pivot table

In [44]:
df_pivot = df_merged_all.pivot_table(index=["client_id" , "visitor_id", "visit_id"], columns="process_step", values="date_time" , aggfunc="max")
#aggfunc - it takes only the *first* time stamp that the step was done
steps = ["start", "step_1", "step_2", "step_3", "confirm"]

df_pivot = df_pivot.reindex(columns=steps)

df_pivot[["start", "step_1", "step_2", "step_3", "confirm"]] = df_pivot[["start", "step_1", "step_2", "step_3", "confirm"]].apply(pd.to_datetime)

df_final_pivot = df_pivot.reset_index()

display(df_final_pivot)
display(df_final_pivot.dtypes)

process_step,client_id,visitor_id,visit_id,start,step_1,step_2,step_3,confirm
0,169,201385055_71273495308,749567106_99161211863_557568,2017-04-12 20:19:36,2017-04-12 20:19:45,2017-04-12 20:20:31,2017-04-12 20:22:05,2017-04-12 20:23:09
1,555,402506806_56087378777,637149525_38041617439_716659,2017-04-15 12:57:56,2017-04-15 12:58:03,2017-04-15 12:58:35,2017-04-15 13:00:14,2017-04-15 13:00:34
2,647,66758770_53988066587,40369564_40101682850_311847,2017-04-12 15:41:28,2017-04-12 15:41:35,2017-04-12 15:41:53,2017-04-12 15:45:02,2017-04-12 15:47:45
3,722,919259913_64837298108,984487154_55831795985_521110,2017-04-19 14:56:16,2017-04-19 14:56:18,2017-04-19 14:56:37,2017-04-19 14:57:27,2017-04-19 15:00:09
4,1028,42237450_62128060588,557292053_87239438319_391157,2017-04-08 18:51:28,2017-04-08 19:00:26,2017-04-08 19:00:17,2017-04-08 18:58:04,NaT
...,...,...,...,...,...,...,...,...
88678,9999729,604429154_69247391147,99583652_41711450505_426179,2017-04-05 13:40:49,2017-04-05 13:41:04,NaT,NaT,NaT
88679,9999729,834634258_21862004160,870243567_56915814033_814203,2017-05-08 16:08:25,2017-05-08 16:08:30,2017-05-08 16:08:40,2017-05-08 16:09:19,2017-05-08 16:09:40
88680,9999729,843385170_36953471821,493310979_9209676464_421146,2017-04-20 14:28:57,2017-04-20 14:22:49,2017-04-20 14:27:36,NaT,NaT
88681,9999832,145538019_54444341400,472154369_16714624241_585315,2017-05-16 16:46:03,2017-05-16 16:46:11,NaT,NaT,NaT


process_step
client_id              int64
visitor_id               str
visit_id                 str
start         datetime64[us]
step_1        datetime64[us]
step_2        datetime64[us]
step_3        datetime64[us]
confirm       datetime64[us]
dtype: object

In [24]:
df_time = df_final_pivot.copy()

df_time["time_start_step1"] = ( df_time["step_1"] - df_time["start"]).dt.total_seconds()
df_time["time_step1_step2"] = ( df_time["step_2"] - df_time["step_1"]).dt.total_seconds()
df_time["time_step3_step3"] = ( df_time["step_3"] - df_time["step_2"]).dt.total_seconds()
df_time["time_step3_confitm"] = ( df_time["confirm"] - df_time["step_3"]).dt.total_seconds()


df_time["confirmation"] = df_time["confirm"].notnull()

time_cols = [ "time_start_step1", "time_step1_step2", "time_step3_step3", "time_step3_confitm",]

display(df_time.head(10))
print(f'Shape:{df_time.shape}')

process_step,client_id,visitor_id,visit_id,start,step_1,step_2,step_3,confirm,time_start_step1,time_step1_step2,time_step3_step3,time_step3_confitm,confirmation
0,169,201385055_71273495308,749567106_99161211863_557568,2017-04-12 20:19:36,2017-04-12 20:19:45,2017-04-12 20:20:31,2017-04-12 20:22:05,2017-04-12 20:23:09,9.0,46.0,94.0,64.0,True
1,555,402506806_56087378777,637149525_38041617439_716659,2017-04-15 12:57:56,2017-04-15 12:58:03,2017-04-15 12:58:35,2017-04-15 13:00:14,2017-04-15 13:00:34,7.0,32.0,99.0,20.0,True
2,647,66758770_53988066587,40369564_40101682850_311847,2017-04-12 15:41:28,2017-04-12 15:41:35,2017-04-12 15:41:53,2017-04-12 15:45:02,2017-04-12 15:47:45,7.0,18.0,189.0,163.0,True
3,722,919259913_64837298108,984487154_55831795985_521110,2017-04-19 14:56:16,2017-04-19 14:56:18,2017-04-19 14:56:37,2017-04-19 14:57:27,2017-04-19 15:00:09,2.0,19.0,50.0,162.0,True
4,1028,42237450_62128060588,557292053_87239438319_391157,2017-04-08 18:51:28,2017-04-08 19:00:26,2017-04-08 19:00:17,2017-04-08 18:58:04,NaT,538.0,-9.0,-133.0,NaN,False
5,1104,194240915_18158000533,543158812_46395476577_767725,2017-06-12 07:49:18,NaT,NaT,NaT,NaT,NaN,NaN,NaN,NaN,False
6,1104,194240915_18158000533,643221571_99977972121_69283,2017-06-20 22:31:33,NaT,NaT,NaT,NaT,NaN,NaN,NaN,NaN,False
7,1186,446844663_31615102958,507052512_11309370126_442139,2017-04-08 15:59:16,NaT,NaT,NaT,NaT,NaN,NaN,NaN,NaN,False
8,1186,446844663_31615102958,795373564_99931517312_810896,2017-04-08 18:05:02,2017-04-08 18:05:13,2017-04-08 18:05:24,NaT,NaT,11.0,11.0,NaN,NaN,False
9,1195,766842522_69992551638,393817425_39015278493_996341,2017-04-05 20:15:26,2017-04-05 20:15:59,2017-04-05 20:17:37,2017-04-05 20:18:08,2017-04-05 20:19:31,33.0,98.0,31.0,83.0,True


Shape:(88683, 13)


In [22]:
df_pivot_time_cleaned = df_time.dropna(subset=['start', 'step_1', 'step_2', 'step_3', 'confirm'], how='all')
print(f'Shape:{df_pivot_time_cleaned.shape}')

Shape:(88683, 13)


### 5.3 client_id who stopped at START

Clients whose furthest reached step was `start`.

In [25]:
STEP_ORDER = {'start': 0, 'step_1': 1, 'step_2': 2, 'step_3': 3, 'confirm': 4}

df_merged_all['step_order'] = df_merged_all['process_step'].map(STEP_ORDER)

max_step = (
    df_merged_all.groupby('client_id')['step_order']
    .max()
    .reset_index()
    .rename(columns={'step_order': 'max_step_order'})
)

clients_at_start = max_step[max_step['max_step_order'] == 0]
print(f'client_id who stopped at START: {len(clients_at_start)}')

client_id who stopped at START: 5461


### 5.4–5.6 client_id who stopped at step_1, step_2, step_3 + time spent at each step

Time elapsed **within each visit_id** between consecutive events.

In [31]:
df_sorted = df_merged_all.sort_values(['visit_id', 'date_time'])

df_sorted['time_at_step'] = (
    df_sorted.groupby('visit_id')['date_time']
    .diff()
    .shift(-1)   
)

def analyse_step(step_name, step_order_num):
    """Clients whose furthest step is step_name, plus mean/median time spent there."""
    step_ids = max_step[max_step['max_step_order'] == step_order_num]['client_id']
    n = len(step_ids)

    times = (
        df_sorted[
            (df_sorted['client_id'].isin(step_ids)) &
            (df_sorted['process_step'] == step_name)
        ]['time_at_step']
        .dropna()
    )

    if not times.empty:
        mean_t   = times.mean()
        median_t = times.median()
        mean_str   = f'{mean_t.seconds // 60}m {mean_t.seconds % 60}s'
        median_str = f'{median_t.seconds // 60}m {median_t.seconds % 60}s'
    else:
        mean_str = median_str = 'N/A'

    print(f'{step_name.upper()}')
    print(f'  client_id who stopped here: {n}')
    print(f'  Mean time at step:          {mean_str}')
    print(f'  Median time at step:        {median_str}')
 

analyse_step('step_1', 1)
analyse_step('step_2', 2)
analyse_step('step_3', 3)
analyse_step('confirm', 4)

STEP_1
  client_id who stopped here: 4689
  Mean time at step:          2m 28s
  Median time at step:        0m 53s
STEP_2
  client_id who stopped here: 3983
  Mean time at step:          2m 9s
  Median time at step:        0m 48s
STEP_3
  client_id who stopped here: 9008
  Mean time at step:          2m 58s
  Median time at step:        1m 18s
CONFIRM
  client_id who stopped here: 44593
  Mean time at step:          4m 14s
  Median time at step:        1m 43s


### 5.7 Of those who completed the process: how many are Test vs Control?

In [33]:
variation_per_client = (
    df_merged_all[['client_id', 'variation']]
    .drop_duplicates(subset='client_id')
)

completed_with_var = completed_clients.merge(variation_per_client, on='client_id')

print('Clients who COMPLETED the process:')
print(completed_with_var['variation'].value_counts().to_string())

Clients who COMPLETED the process:
variation
Test       14925
Control    12771


In [34]:
completed_ids = set(completed_clients['client_id'])
all_clients = variation_per_client.copy()

not_completed_with_var = all_clients[
    ~all_clients['client_id'].isin(completed_ids)
]

print('Clients who did NOT complete the process:')
print(not_completed_with_var['variation'].value_counts().to_string())

Clients who did NOT complete the process:
variation
Test       11007
Control     9755


## 6. Completion time insights: Test vs Control

## 7. 

In [63]:
import scipy.stats as stats

In [65]:
print("Available columns in df_merged_all:")
print(df_merged_all.columns.tolist())
print("\nAvailable columns in df_time:")
print(df_time.columns.tolist())

df_time_with_var = df_time.merge(df_clients[['client_id', 'variation']], on='client_id', how='inner')

time_steps = ["time_start_step1", "time_step1_step2", "time_step3_step3", "time_step3_confitm"]

print("=== Mean time within the same group ===")
for var in ['Control', 'Test']:
    print(f"\nVariation: {var}")
    group_df = df_time_with_var[df_time_with_var['variation'] == var]
    for step in time_steps:
        mean_time = group_df[step].mean()
        print(f"  Mean {step}: {mean_time:.2f} seconds")

print("\n" + "="*50 + "\n")

print("=== T-student / comparison between both groups ===")
control_group = df_time_with_var[df_time_with_var['variation'] == 'Control']
test_group = df_time_with_var[df_time_with_var['variation'] == 'Test']

for step in time_steps:
    c_times = control_group[step].dropna()
    t_times = test_group[step].dropna()
    
    t_stat, p_val = stats.ttest_ind(c_times, t_times, equal_var=False)
    
    print(f"\nStep: {step}")
    print(f"  P-value: {p_val}")
    if p_val < 0.05:
        print("  Yes, there is a statistical diference between test and control")
    else:
        print("  No, there is no significance. Statistically times are the same.")

Available columns in df_merged_all:
['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time', 'variation', 'step_order']

Available columns in df_time:
['client_id', 'visitor_id', 'visit_id', 'start', 'step_1', 'step_2', 'step_3', 'confirm', 'time_start_step1', 'time_step1_step2', 'time_step3_step3', 'time_step3_confitm', 'confirmation']
=== Mean time within the same group ===

Variation: Control
  Mean time_start_step1: 38.72 seconds
  Mean time_step1_step2: 33.30 seconds
  Mean time_step3_step3: 85.48 seconds
  Mean time_step3_confitm: 127.74 seconds

Variation: Test
  Mean time_start_step1: 25.58 seconds
  Mean time_step1_step2: 35.59 seconds
  Mean time_step3_step3: 79.81 seconds
  Mean time_step3_confitm: 102.68 seconds


=== T-student / comparison between both groups ===

Step: time_start_step1
  P-value: 9.723011066535274e-09
  Yes, there is a statistical diference between test and control

Step: time_step1_step2
  P-value: 0.15856640741467254
  No, there is no signif